# Dissertation — Phase 2A RS-1 Historical-Origin Final Recovery Refit

**Stage:** RS-4H — one-time Historical-origin recovery refit only.

Run this notebook in a **fresh Kaggle Tesla T4 runtime**, after the Full RS-4F
artifact ZIP has been downloaded and independently SHA-256 verified.

The verified Full artifact is frozen and must never be retrained.

## Historical recovery contract

- Historical N = **5,602**
- Physical batch = **2**
- Gradient accumulation = **16**
- Effective batch = **32**
- Execute **3 epochs only**
- Optimizer updates / epoch = **176**
- Executed optimizer updates = **528**
- Scheduler horizon = **8 epochs / 1,408 steps**
- Warmup = **140**
- FP32 only; no autocast; no GradScaler
- No validation and no held-out inference during refit

**Critical rule:** train 3 epochs while retaining the frozen 8-epoch scheduler
trajectory. A 3-epoch scheduler is prohibited.

Previously verified provenance:

- Full RS-4F ZIP SHA-256:
  `12962bd779ab9523d24736af2c68823ac66d41c56334261d7341d31991c8e545`
- Cell 3D authoritative registry SHA-256:
  `8848587a2dd24976710a3f7a30514ed3a55f24228178f470c98cd64a1645c118`
- RS-2 protocol SHA-256:
  `a6c2ae70b0b348544017ebd8aeda0ad8ecb95a54fc4c571319090d6a4e29dc9f`
- RS-3S smoke SHA-256:
  `fd8de543e69e7810ec5cf0b7b6b0c1b9a0cadd2bb7717af0a73af2f8de451b1d`
- RS-3S initial whole-model SHA-256:
  `c66e8175c2e9f031da9bfd766479b4fceed74caa1c47888bf0cdc0f9d15b01d1`
- RS-3P memory-contract SHA-256:
  `519114f014063354336294ae6a96d1385974d355d9f4169739ef78c9223a5b53`

### Run order

Run **H0 → H1 → H2 → H3** only.


In [ ]:
# =============================================================================
# H0 — FRESH RUNTIME / DETERMINISM / PATHS / ATOMIC UTILITIES
# NO TRAINING | NO HELD-OUT ACCESS
# =============================================================================

import os

# Set before torch import / CUDA work.
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import gc
import json
import math
import time
import random
import hashlib
import platform
import tempfile
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import transformers

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

RECOVERY_ID = "RS1"
STAGE_ID = "RS-4H"
SEED = 42

WORK_ROOT = Path("/kaggle/working/phase2A_RS1_HISTORICAL")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def resolve_unique_kaggle_input(filename):
    root = Path("/kaggle/input")
    matches = sorted(p for p in root.rglob(filename) if p.is_file())
    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not locate {filename!r} under /kaggle/input. "
            "Attach the Yelp training dataset."
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Found multiple files named {filename!r}. "
            "Remove duplicate inputs before training.\n"
            + "\n".join(str(p) for p in matches)
        )
    return matches[0]

TRAIN_PATH = resolve_unique_kaggle_input("new_data_train.csv")

# This final-refit notebook deliberately never resolves/opens new_data_test.csv.
ALLOW_MODEL_TRAINING = False
ALLOW_HELDOUT_REPREDICTION = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def canonical_json_sha(payload):
    raw = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def file_sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def atomic_write_json(payload, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(
        dir=str(destination.parent),
        prefix=destination.name + ".tmp.",
        suffix=".json",
    )
    os.close(fd)
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, destination)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

def atomic_write_csv(df, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(
        dir=str(destination.parent),
        prefix=destination.name + ".tmp.",
        suffix=".csv",
    )
    os.close(fd)
    try:
        df.to_csv(tmp, index=False)
        with open(tmp, "rb") as f:
            os.fsync(f.fileno())
        os.replace(tmp, destination)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

def atomic_torch_save(payload, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(
        dir=str(destination.parent),
        prefix=destination.name + ".tmp.",
        suffix=".pt",
    )
    os.close(fd)
    try:
        with open(tmp, "wb") as f:
            torch.save(payload, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, destination)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

assert torch.cuda.is_available(), (
    "A GPU is required. Enable a Kaggle GPU accelerator before continuing."
)
assert "T4" in torch.cuda.get_device_name(0), (
    "RS-4F memory contract was validated on Tesla T4. "
    f"Current GPU: {torch.cuda.get_device_name(0)}"
)

ENVIRONMENT = {
    "recovery_id": RECOVERY_ID,
    "stage": STAGE_ID,
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "seed": SEED,
    "deterministic_algorithms": True,
    "tf32": False,
    "cublas_workspace_config": os.environ["CUBLAS_WORKSPACE_CONFIG"],
    "heldout_file_opened": False,
}
ENVIRONMENT["payload_sha256"] = canonical_json_sha(ENVIRONMENT)
atomic_write_json(
    ENVIRONMENT,
    WORK_ROOT / "phase2A_RS1_HISTORICAL_environment.json",
)

print("=" * 96)
print("H0 — RS-4H FRESH RUNTIME")
print("=" * 96)
print(f"Python       : {ENVIRONMENT['python']}")
print(f"PyTorch      : {ENVIRONMENT['torch']}")
print(f"Transformers : {ENVIRONMENT['transformers']}")
print(f"GPU          : {ENVIRONMENT['gpu']}")
print(f"Train file   : {TRAIN_PATH}")
print(f"Work root    : {WORK_ROOT}")
print(f"Environment SHA256: {ENVIRONMENT['payload_sha256']}")
print("\n✓ H0 PASSED — no held-out file was opened")


In [ ]:
# =============================================================================
# H1 — TRAINING-ORIGIN ONLY RECONSTRUCTION + CRYPTOGRAPHIC AUDIT
# NO TEST FILE | NO HELD-OUT REGIMES | NO TRAINING
# =============================================================================

import csv
import io

TARGET = "flagged"
TEXT_COL = "reviewContent"
TEMPORAL_CUTOFF = pd.Timestamp("2011-01-01")

EXPECTED_FULL_ORIGIN_FP = (
    "6539dc98b3517bd778df0d0b4abba1f4beb3f9120ce70bdaa1be540404543195"
)
EXPECTED_HIST_ORIGIN_FP = (
    "ed4e3a55e0c52f8949873847ba44ac8bebe8de77efd96daacc4cb4c5e592ea2e"
)
FP_COLS = ["reviewID", "reviewerID", "flagged", "reviewContent"]

EXPECTED_COLUMNS = [
    "reviewID", "reviewerID", "restaurantID", "date", "rating",
    "reviewUsefulCount", "reviewContent", "flagged", "name", "location",
    "yelpJoinDate", "friendCount", "reviewCount", "firstCount",
    "usefulCount", "coolCount", "funnyCount", "complimentCount",
    "tipCount", "fanCount", "restaurantRating", "ReviewLength",
]

def _read_native_tsv(path):
    return pd.read_csv(path, sep="\t", index_col=0)

def _repair_known_comma_wrapper_export(path):
    reconstructed_lines = []
    wrapper_widths = []

    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f, delimiter=",", quotechar='"')
        for physical_line_no, row in enumerate(reader, start=1):
            if not row:
                raise RuntimeError(
                    f"Unexpected blank wrapper row at line {physical_line_no}"
                )

            wrapper_widths.append(len(row))
            last = len(row)
            while last > 1 and row[last - 1] == "":
                last -= 1

            reconstructed = ",".join(row[:last])

            if reconstructed.count("\t") != 22:
                raise RuntimeError(
                    "Wrapper recovery failed structural validation.\n"
                    f"Physical line: {physical_line_no}\n"
                    f"Recovered tab count: {reconstructed.count(chr(9))}"
                )
            reconstructed_lines.append(reconstructed)

    repaired_text = "\n".join(reconstructed_lines) + "\n"
    df = pd.read_csv(io.StringIO(repaired_text), sep="\t", index_col=0)

    return df, {
        "serialization": "comma_csv_wrapper_repaired_to_original_tsv",
        "physical_rows": int(len(reconstructed_lines)),
        "wrapper_min_fields": int(min(wrapper_widths)),
        "wrapper_max_fields": int(max(wrapper_widths)),
        "rows_skipped": 0,
    }

def load_train_strict(path):
    try:
        df = _read_native_tsv(path)
        meta = {
            "serialization": "native_tsv",
            "rows_skipped": 0,
        }
    except pd.errors.ParserError as native_error:
        with open(path, "r", encoding="utf-8", errors="strict") as f:
            header = f.readline()

        wrapper_signature = (
            header.count("\t") == 22
            and header.count(",") > 0
            and header.startswith('"')
        )
        if not wrapper_signature:
            raise RuntimeError(
                "Strict TSV parsing failed and the source does not match the "
                "audited comma-wrapper signature. No fallback is permitted."
            ) from native_error

        print("⚠ Known comma-wrapper serialization detected.")
        print("  Applying audited lossless recovery; zero rows skipped.")
        df, meta = _repair_known_comma_wrapper_export(path)

    assert list(df.columns) == EXPECTED_COLUMNS, (
        f"Training schema mismatch.\nObserved: {list(df.columns)}"
    )

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    assert df["date"].notna().all()

    meta["path"] = str(path)
    meta["n_rows_loaded"] = int(len(df))
    meta["n_columns_loaded"] = int(len(df.columns))
    return df, meta

def content_fingerprint(df):
    for col in FP_COLS:
        assert col in df.columns

    assert df["reviewID"].notna().all()
    assert df["reviewerID"].notna().all()
    assert df["reviewContent"].notna().all()
    assert df["flagged"].isin([0, 1]).all()

    x = df[FP_COLS].copy()
    x["reviewID"] = x["reviewID"].astype(str)
    x["reviewerID"] = x["reviewerID"].astype(str)
    x["flagged"] = x["flagged"].astype(int)
    x["reviewContent"] = x["reviewContent"].astype(str)
    x = x.sort_values("reviewID", kind="mergesort").reset_index(drop=True)

    payload = json.dumps(
        x.to_dict(orient="records"),
        separators=(",", ":"),
        ensure_ascii=False,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

train_raw, TRAIN_LOAD_META = load_train_strict(TRAIN_PATH)

assert len(train_raw) == 9929
assert train_raw[TEXT_COL].isna().sum() == 3

train_full = (
    train_raw.loc[train_raw[TEXT_COL].notna()]
    .copy()
    .reset_index(drop=True)
)

train_temporal = (
    train_full.loc[train_full["date"] < TEMPORAL_CUTOFF]
    .copy()
    .reset_index(drop=True)
)

assert len(train_full) == 9926
assert (train_full[TARGET] == 0).sum() == 4993
assert (train_full[TARGET] == 1).sum() == 4933

assert len(train_temporal) == 5602
assert (train_temporal[TARGET] == 0).sum() == 2756
assert (train_temporal[TARGET] == 1).sum() == 2846

FULL_FP = content_fingerprint(train_full)
HIST_FP = content_fingerprint(train_temporal)

assert FULL_FP == EXPECTED_FULL_ORIGIN_FP
assert HIST_FP == EXPECTED_HIST_ORIGIN_FP

TRAIN_MANIFEST = {
    "recovery_id": RECOVERY_ID,
    "stage": STAGE_ID,
    "heldout_loaded": False,
    "train_raw_n": int(len(train_raw)),
    "full_n": int(len(train_full)),
    "historical_n": int(len(train_temporal)),
    "full_origin_fingerprint": FULL_FP,
    "historical_origin_fingerprint": HIST_FP,
    "source_serialization": TRAIN_LOAD_META,
}
TRAIN_MANIFEST["payload_sha256"] = canonical_json_sha(TRAIN_MANIFEST)

atomic_write_json(
    TRAIN_MANIFEST,
    WORK_ROOT / "phase2A_RS1_HISTORICAL_training_origin_manifest.json",
)

print("=" * 96)
print("H1 — TRAINING-ORIGIN AUDIT")
print("=" * 96)
print(f"Raw train        : {len(train_raw):,}")
print(f"Full origin      : {len(train_full):,} | {FULL_FP}")
print(f"Historical origin: {len(train_temporal):,} | {HIST_FP}")
print(f"Rows skipped     : {TRAIN_LOAD_META['rows_skipped']}")
print(f"Manifest SHA256  : {TRAIN_MANIFEST['payload_sha256']}")
print("\n✓ H1 PASSED — exact Full/Historical origins recovered")
print("✓ No held-out test/regime data were loaded")


In [ ]:
# =============================================================================
# H2 — FROZEN HISTORICAL SCIENTIFIC + PERMANENT MEMORY CONTRACT
# NO TRAINING | NO HELD-OUT ACCESS
# =============================================================================

MODEL_CHECKPOINT = "microsoft/deberta-v3-base"
MODEL_REVISION = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
MAX_LENGTH = 256
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

AUTH1_PHYSICAL_BATCH = 16
AUTH1_GRAD_ACCUM = 2
AUTH1_EFFECTIVE_BATCH = 32

RS1_PHYSICAL_BATCH = 2
RS1_GRAD_ACCUM = 16
RS1_EFFECTIVE_BATCH = 32

HIST_EPOCHS_EXECUTED = 3
HIST_SCHEDULER_HORIZON_EPOCHS = 8
HIST_UPDATES_PER_EPOCH = 176
HIST_EXECUTED_UPDATES = 528
HIST_SCHEDULER_TOTAL_STEPS = 1408
HIST_WARMUP_STEPS = 140

CELL3D_AUTHORITATIVE_SHA256 = (
    "8848587a2dd24976710a3f7a30514ed3a55f24228178f470c98cd64a1645c118"
)
RS2_PROTOCOL_SHA256 = (
    "a6c2ae70b0b348544017ebd8aeda0ad8ecb95a54fc4c571319090d6a4e29dc9f"
)
RS3S_SMOKE_SHA256 = (
    "fd8de543e69e7810ec5cf0b7b6b0c1b9a0cadd2bb7717af0a73af2f8de451b1d"
)
RS3S_EXPECTED_INITIAL_MODEL_SHA256 = (
    "c66e8175c2e9f031da9bfd766479b4fceed74caa1c47888bf0cdc0f9d15b01d1"
)
RS3P_MEMORY_CONTRACT_SHA256 = (
    "519114f014063354336294ae6a96d1385974d355d9f4169739ef78c9223a5b53"
)
FULL_RS4F_ZIP_SHA256 = (
    "12962bd779ab9523d24736af2c68823ac66d41c56334261d7341d31991c8e545"
)

assert len(train_temporal) == 5602
assert HIST_FP == EXPECTED_HIST_ORIGIN_FP

assert RS1_PHYSICAL_BATCH == 2
assert RS1_GRAD_ACCUM == 16
assert RS1_EFFECTIVE_BATCH == AUTH1_EFFECTIVE_BATCH == 32

microbatches_per_epoch = math.ceil(len(train_temporal) / RS1_PHYSICAL_BATCH)
updates_per_epoch = math.ceil(microbatches_per_epoch / RS1_GRAD_ACCUM)

assert microbatches_per_epoch == 2801
assert updates_per_epoch == HIST_UPDATES_PER_EPOCH == 176
assert updates_per_epoch * HIST_EPOCHS_EXECUTED == HIST_EXECUTED_UPDATES == 528

full_groups = len(train_temporal) // RS1_EFFECTIVE_BATCH
partial_rows = len(train_temporal) % RS1_EFFECTIVE_BATCH
assert full_groups == 175
assert partial_rows == 2

# CRITICAL: scheduler horizon is 8 epochs even though execution stops at 3.
assert HIST_SCHEDULER_TOTAL_STEPS == HIST_UPDATES_PER_EPOCH * 8
assert HIST_SCHEDULER_TOTAL_STEPS == 1408
assert HIST_WARMUP_STEPS == 140
assert HIST_EXECUTED_UPDATES < HIST_SCHEDULER_TOTAL_STEPS

REFIT_CONTRACT = {
    "recovery_id": RECOVERY_ID,
    "stage": STAGE_ID,
    "origin": "Historical",
    "auth1_primary_preserved_separately": True,
    "claim_bitwise_auth1_replication": False,
    "full_rs4f_artifact_verified_before_historical": True,
    "full_rs4f_zip_sha256": FULL_RS4F_ZIP_SHA256,
    "model_checkpoint": MODEL_CHECKPOINT,
    "model_revision": MODEL_REVISION,
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_grad_norm": MAX_GRAD_NORM,
    "epochs_executed": HIST_EPOCHS_EXECUTED,
    "scheduler_horizon_epochs": HIST_SCHEDULER_HORIZON_EPOCHS,
    "optimizer": "torch.optim.AdamW",
    "precision": "FP32",
    "autocast": False,
    "grad_scaler": False,
    "auth1_effective_batch": {
        "physical_batch": AUTH1_PHYSICAL_BATCH,
        "gradient_accumulation": AUTH1_GRAD_ACCUM,
        "effective_batch": AUTH1_EFFECTIVE_BATCH,
    },
    "rs1_memory_safe_batch": {
        "physical_batch": RS1_PHYSICAL_BATCH,
        "gradient_accumulation": RS1_GRAD_ACCUM,
        "effective_batch": RS1_EFFECTIVE_BATCH,
        "larger_physical_batch_permitted": False,
    },
    "microbatches_per_epoch": microbatches_per_epoch,
    "optimizer_updates_per_epoch": updates_per_epoch,
    "executed_optimizer_updates": HIST_EXECUTED_UPDATES,
    "scheduler_total_steps": HIST_SCHEDULER_TOTAL_STEPS,
    "warmup_steps": HIST_WARMUP_STEPS,
    "full_groups_of_32": full_groups,
    "final_partial_group_rows": partial_rows,
    "dataloader_num_workers": 0,
    "pin_memory": False,
    "validation_during_refit": False,
    "heldout_inference_during_refit": False,
    "resume_checkpoint_frequency": "epoch_boundary",
    "prior_governance": {
        "cell3d_sha256": CELL3D_AUTHORITATIVE_SHA256,
        "rs2_protocol_sha256": RS2_PROTOCOL_SHA256,
        "rs3s_smoke_sha256": RS3S_SMOKE_SHA256,
        "rs3p_memory_contract_sha256": RS3P_MEMORY_CONTRACT_SHA256,
    },
}
REFIT_CONTRACT["payload_sha256"] = canonical_json_sha(REFIT_CONTRACT)

atomic_write_json(
    REFIT_CONTRACT,
    WORK_ROOT / "phase2A_RS1_HISTORICAL_refit_contract.json",
)

print("=" * 96)
print("H2 — HISTORICAL REFIT CONTRACT")
print("=" * 96)
print(f"Model revision           : {MODEL_REVISION}")
print(f"Historical N             : {len(train_temporal):,}")
print(f"Physical batch           : {RS1_PHYSICAL_BATCH}")
print(f"Gradient accumulation    : {RS1_GRAD_ACCUM}")
print(f"Effective batch          : {RS1_EFFECTIVE_BATCH}")
print(f"Microbatches/epoch       : {microbatches_per_epoch}")
print(f"Updates/epoch            : {updates_per_epoch}")
print(f"Epochs executed          : {HIST_EPOCHS_EXECUTED}")
print(f"Executed updates         : {HIST_EXECUTED_UPDATES}")
print(f"Scheduler horizon epochs : {HIST_SCHEDULER_HORIZON_EPOCHS}")
print(f"Scheduler total steps    : {HIST_SCHEDULER_TOTAL_STEPS}")
print(f"Warmup steps             : {HIST_WARMUP_STEPS}")
print(f"Contract SHA256          : {REFIT_CONTRACT['payload_sha256']}")
print("\n✓ H2 PASSED")
print("✓ Historical stops after epoch 3 under the frozen 8-epoch scheduler")
print("✓ Physical batch > 2 is prohibited in RS-4H")


In [ ]:
# =============================================================================
# H3 — RS-4H ONE-TIME HISTORICAL-ORIGIN FINAL RECOVERY REFIT
# MEMORY-SAFE 2×16 | FP32 | CHECKPOINT/RESUME | ATOMIC FINAL BUNDLE
# NO VALIDATION | NO HELD-OUT ACCESS | NO TUNING
# =============================================================================

import torch.nn as nn

FINAL_DIR = WORK_ROOT / "phase2A_RS1_HISTORICAL_ARTIFACT"
STAGING_DIR = WORK_ROOT / "phase2A_RS1_HISTORICAL_ARTIFACT.staging"
RESUME_PATH = WORK_ROOT / "phase2A_RS1_HISTORICAL_resume_checkpoint.pt"
FAILURE_PATH = WORK_ROOT / "phase2A_RS1_HISTORICAL_failure.json"
ZIP_PATH = WORK_ROOT / "phase2A_RS1_HISTORICAL_ARTIFACT.zip"
ZIP_SHA_PATH = WORK_ROOT / "phase2A_RS1_HISTORICAL_ARTIFACT.zip.sha256.txt"
READY_PATH = WORK_ROOT / "phase2A_RS1_HISTORICAL_READY_TO_EXPORT.txt"

# ------------------------- hard rerun / overwrite guards ----------------------
if FINAL_DIR.exists() or ZIP_PATH.exists():
    raise RuntimeError(
        "A completed/final RS-4F artifact already exists. "
        "This one-time refit will not overwrite or retrain it."
    )
if STAGING_DIR.exists():
    raise RuntimeError(
        f"Stale staging directory exists:\n{STAGING_DIR}\n"
        "Do not delete it blindly. Inspect it before another finalisation attempt."
    )

assert ALLOW_MODEL_TRAINING is False
assert ALLOW_HELDOUT_REPREDICTION is False
assert RS1_PHYSICAL_BATCH == 2
assert RS1_GRAD_ACCUM == 16
assert RS1_EFFECTIVE_BATCH == 32
assert HIST_UPDATES_PER_EPOCH == 176
assert HIST_EXECUTED_UPDATES == 528
assert HIST_SCHEDULER_TOTAL_STEPS == 1408
assert HIST_WARMUP_STEPS == 140
assert HIST_FP == EXPECTED_HIST_ORIGIN_FP

# ------------------------------ GPU hygiene ----------------------------------
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

free_bytes, total_bytes = torch.cuda.mem_get_info()
free_gib = free_bytes / (1024 ** 3)
total_gib = total_bytes / (1024 ** 3)

print("=" * 96)
print("H3 / RS-4H — HISTORICAL FINAL RECOVERY REFIT")
print("=" * 96)
print(f"GPU total memory : {total_gib:.2f} GiB")
print(f"GPU free memory  : {free_gib:.2f} GiB")

# With a fresh T4 runtime, materially less than 8 GiB free means some unrelated
# GPU allocation survived. Abort before training rather than inviting OOM.
assert free_gib >= 8.0, (
    f"Only {free_gib:.2f} GiB GPU memory is free. "
    "Restart the Kaggle runtime and run H0→H3 again. "
    "Do NOT increase/decrease the frozen batch contract."
)

# ------------------------------ tokenizer ------------------------------------
reset_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    revision=MODEL_REVISION,
    use_fast=True,
)

assert type(tokenizer).__name__ == "DebertaV2Tokenizer"
assert int(tokenizer.vocab_size) == 128000

probe_ids = tokenizer(
    "determinism probe",
    add_special_tokens=True,
    truncation=True,
)["input_ids"]
assert list(probe_ids) == [78306, 8704]

# ------------------------- CPU pre-tokenisation -------------------------------
# Same tokenizer arguments as the historical/recovery pipeline, performed once
# on CPU to prevent tokenisation from becoming the throughput bottleneck.
texts = train_temporal[TEXT_COL].astype(str).tolist()
labels_cpu = torch.tensor(
    train_temporal[TARGET].astype(int).to_numpy(),
    dtype=torch.long,
)

encoded_cpu = tokenizer(
    texts,
    add_special_tokens=True,
    max_length=MAX_LENGTH,
    truncation=True,
    padding="max_length",
    return_attention_mask=True,
    return_tensors="pt",
)

assert encoded_cpu["input_ids"].shape == (len(train_temporal), MAX_LENGTH)
assert encoded_cpu["attention_mask"].shape == (len(train_temporal), MAX_LENGTH)

# Batch-tokenisation equivalence audit against per-item tokenisation.
probe_rows = [0, 1, 17, 101, 999, len(train_temporal) - 1]
for idx in probe_rows:
    single = tokenizer(
        texts[idx],
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_attention_mask=True,
        return_tensors="pt",
    )
    assert torch.equal(
        single["input_ids"].squeeze(0),
        encoded_cpu["input_ids"][idx],
    )
    assert torch.equal(
        single["attention_mask"].squeeze(0),
        encoded_cpu["attention_mask"][idx],
    )

def tensor_bundle_sha(encoded, labels):
    h = hashlib.sha256()
    for key in sorted(encoded.keys()):
        t = encoded[key].detach().cpu().contiguous()
        h.update(key.encode("utf-8"))
        h.update(str(tuple(t.shape)).encode("utf-8"))
        h.update(str(t.dtype).encode("utf-8"))
        h.update(t.numpy().tobytes(order="C"))
    t = labels.detach().cpu().contiguous()
    h.update(b"labels")
    h.update(str(tuple(t.shape)).encode("utf-8"))
    h.update(str(t.dtype).encode("utf-8"))
    h.update(t.numpy().tobytes(order="C"))
    return h.hexdigest()

TOKENIZATION_SHA = tensor_bundle_sha(encoded_cpu, labels_cpu)
print(f"Tokenization SHA256: {TOKENIZATION_SHA}")

class PretokenizedDataset(Dataset):
    def __init__(self, encoded, labels):
        self.encoded = {k: v.cpu() for k, v in encoded.items()}
        self.labels = labels.cpu()
        self.n = int(len(labels))

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encoded.items()}
        item["labels"] = self.labels[idx]
        item["_row_idx"] = torch.tensor(idx, dtype=torch.long)
        return item

dataset = PretokenizedDataset(encoded_cpu, labels_cpu)

shuffle_gen = torch.Generator()
shuffle_gen.manual_seed(SEED)

loader = DataLoader(
    dataset,
    batch_size=RS1_PHYSICAL_BATCH,
    shuffle=True,
    generator=shuffle_gen,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
)

assert len(loader) == 2801

# ------------------------ model-loading safeguards ----------------------------
def nonfinite_parameter_names(model):
    bad = []
    with torch.no_grad():
        for name, param in model.named_parameters():
            if not torch.isfinite(param).all().item():
                bad.append(name)
    return bad

def initialise_task_specific_layers(model):
    # Exact RS-3S deterministic task-head initialisation.
    reset_seed(SEED)
    initialised = []
    with torch.no_grad():
        for name, module in model.named_modules():
            if not (
                name.startswith("pooler")
                or name.startswith("classifier")
            ):
                continue
            if isinstance(module, nn.Linear):
                module.weight.normal_(
                    mean=0.0,
                    std=float(model.config.initializer_range),
                )
                if module.bias is not None:
                    module.bias.zero_()
                initialised.append(name)
    assert initialised == ["pooler.dense", "classifier"], initialised
    return initialised

def whole_model_sha256(model):
    h = hashlib.sha256()
    with torch.no_grad():
        for name, param in model.named_parameters():
            arr = (
                param.detach()
                .to(dtype=torch.float32, device="cpu")
                .contiguous()
                .numpy()
            )
            h.update(name.encode("utf-8"))
            h.update(str(tuple(arr.shape)).encode("utf-8"))
            h.update(arr.tobytes(order="C"))
    return h.hexdigest()

reset_seed(SEED)
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    revision=MODEL_REVISION,
    num_labels=2,
    dtype=torch.float32,
).to(device=DEVICE, dtype=torch.float32)

bad_before = nonfinite_parameter_names(model)
disallowed_bad = [
    x for x in bad_before
    if not (x.startswith("pooler.") or x.startswith("classifier."))
]
assert not disallowed_bad, (
    f"Non-finite pretrained-backbone parameters: {disallowed_bad[:20]}"
)

INIT_LAYERS = initialise_task_specific_layers(model)
assert not nonfinite_parameter_names(model)
assert {p.dtype for p in model.parameters()} == {torch.float32}

INITIAL_MODEL_SHA = whole_model_sha256(model)
print(f"Initial whole-model SHA256: {INITIAL_MODEL_SHA}")

assert INITIAL_MODEL_SHA == RS3S_EXPECTED_INITIAL_MODEL_SHA256, (
    "Initial model state differs from the already-passed RS-3S deterministic "
    "smoke gate. Historical refit is blocked."
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

optimizer_contract = {
    "class": type(optimizer).__name__,
    "lr": float(optimizer.param_groups[0]["lr"]),
    "betas": tuple(float(x) for x in optimizer.param_groups[0]["betas"]),
    "eps": float(optimizer.param_groups[0]["eps"]),
    "weight_decay": float(optimizer.param_groups[0]["weight_decay"]),
    "foreach": optimizer.param_groups[0].get("foreach", None),
    "fused": optimizer.param_groups[0].get("fused", None),
}
assert optimizer_contract["class"] == "AdamW"
assert optimizer_contract["lr"] == LEARNING_RATE
assert optimizer_contract["betas"] == (0.9, 0.999)
assert optimizer_contract["eps"] == 1e-8
assert optimizer_contract["weight_decay"] == WEIGHT_DECAY

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=HIST_WARMUP_STEPS,
    num_training_steps=HIST_SCHEDULER_TOTAL_STEPS,
)

assert HIST_EPOCHS_EXECUTED == 3
assert HIST_SCHEDULER_HORIZON_EPOCHS == 8
assert HIST_EXECUTED_UPDATES == 528
assert HIST_SCHEDULER_TOTAL_STEPS == 1408
assert HIST_WARMUP_STEPS == 140
print(
    "Historical scheduler verified: "
    "3 executed epochs under 8-epoch / 1,408-step horizon"
)

# -------------------------- checkpoint / resume -------------------------------
history = []
completed_epoch = 0
total_updates = 0

def save_resume_checkpoint(completed_epoch, total_updates, history):
    payload = {
        "schema": "phase2A_RS1_HISTORICAL_epoch_boundary_resume_v1",
        "stage": STAGE_ID,
        "origin": "Historical",
        "historical_origin_fingerprint": HIST_FP,
        "model_revision": MODEL_REVISION,
        "refit_contract_sha256": REFIT_CONTRACT["payload_sha256"],
        "prior_rs3p_memory_contract_sha256": RS3P_MEMORY_CONTRACT_SHA256,
        "tokenization_sha256": TOKENIZATION_SHA,
        "initial_model_sha256": INITIAL_MODEL_SHA,
        "completed_epoch": int(completed_epoch),
        "total_updates": int(total_updates),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "shuffle_generator_state": shuffle_gen.get_state(),
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_cpu_rng_state": torch.get_rng_state(),
        "torch_cuda_rng_state_all": torch.cuda.get_rng_state_all(),
        "history": history,
    }
    atomic_torch_save(payload, RESUME_PATH)

if RESUME_PATH.exists():
    print("\nExisting epoch-boundary resume checkpoint detected.")
    checkpoint = torch.load(
        RESUME_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    assert checkpoint["schema"] == "phase2A_RS1_HISTORICAL_epoch_boundary_resume_v1"
    assert checkpoint["stage"] == STAGE_ID
    assert checkpoint["origin"] == "Historical"
    assert checkpoint["historical_origin_fingerprint"] == HIST_FP
    assert checkpoint["model_revision"] == MODEL_REVISION
    assert checkpoint["refit_contract_sha256"] == REFIT_CONTRACT["payload_sha256"]
    assert checkpoint["prior_rs3p_memory_contract_sha256"] == RS3P_MEMORY_CONTRACT_SHA256
    assert checkpoint["tokenization_sha256"] == TOKENIZATION_SHA
    assert checkpoint["initial_model_sha256"] == INITIAL_MODEL_SHA

    model.load_state_dict(checkpoint["model_state"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    shuffle_gen.set_state(checkpoint["shuffle_generator_state"])

    random.setstate(checkpoint["python_random_state"])
    np.random.set_state(checkpoint["numpy_random_state"])
    torch.set_rng_state(checkpoint["torch_cpu_rng_state"])
    torch.cuda.set_rng_state_all(checkpoint["torch_cuda_rng_state_all"])

    completed_epoch = int(checkpoint["completed_epoch"])
    total_updates = int(checkpoint["total_updates"])
    history = list(checkpoint["history"])

    assert 0 < completed_epoch < HIST_EPOCHS_EXECUTED
    assert total_updates == completed_epoch * HIST_UPDATES_PER_EPOCH
    assert len(history) == completed_epoch

    print(
        f"✓ Resuming safely after completed epoch {completed_epoch}; "
        f"optimizer update {total_updates}."
    )

start_epoch = completed_epoch + 1

# ------------------------------- TRAIN ---------------------------------------
ALLOW_MODEL_TRAINING = True
torch.cuda.reset_peak_memory_stats()

try:
    for epoch in range(start_epoch, HIST_EPOCHS_EXECUTED + 1):
        torch.cuda.synchronize()
        epoch_t0 = time.time()
        torch.cuda.reset_peak_memory_stats()

        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss_sum = 0.0
        seen = 0
        epoch_updates = 0
        epoch_row_order = []

        n_microbatches = len(loader)

        for step, batch in enumerate(loader, start=1):
            row_idx = batch.pop("_row_idx")
            epoch_row_order.extend(row_idx.tolist())

            batch = {
                k: v.to(DEVICE, non_blocking=False)
                for k, v in batch.items()
            }

            grp_start = ((step - 1) // RS1_GRAD_ACCUM) * RS1_GRAD_ACCUM + 1
            grp_end = min(
                grp_start + RS1_GRAD_ACCUM - 1,
                n_microbatches,
            )
            accum_denom = grp_end - grp_start + 1

            output = model(**batch)
            raw_loss = output.loss

            if not torch.isfinite(raw_loss).item():
                raise FloatingPointError(
                    f"Non-finite loss at epoch={epoch}, microbatch={step}"
                )
            if not torch.isfinite(output.logits).all().item():
                raise FloatingPointError(
                    f"Non-finite logits at epoch={epoch}, microbatch={step}"
                )

            bn = int(batch["labels"].size(0))
            running_loss_sum += float(raw_loss.detach().cpu()) * bn
            seen += bn

            (raw_loss / accum_denom).backward()

            update_boundary = (
                step % RS1_GRAD_ACCUM == 0
                or step == n_microbatches
            )

            if update_boundary:
                # error_if_nonfinite provides a whole-model gradient finite gate
                # without repeatedly materialising full parameter copies.
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM,
                    error_if_nonfinite=True,
                )

                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                epoch_updates += 1
                total_updates += 1

                if epoch_updates % 50 == 0 or epoch_updates == HIST_UPDATES_PER_EPOCH:
                    print(
                        f"  Epoch {epoch}/{HIST_EPOCHS_EXECUTED} | "
                        f"update {epoch_updates:03d}/{HIST_UPDATES_PER_EPOCH} | "
                        f"global {total_updates:04d}/{HIST_EXECUTED_UPDATES} | "
                        f"loss_so_far={running_loss_sum/seen:.4f}"
                    )

            # Free references before the next microbatch.
            del output, raw_loss, batch

        assert seen == len(train_temporal)
        assert epoch_updates == HIST_UPDATES_PER_EPOCH
        assert len(epoch_row_order) == len(train_temporal)
        assert len(set(epoch_row_order)) == len(train_temporal)
        assert sorted(epoch_row_order) == list(range(len(train_temporal)))

        expected_total = epoch * HIST_UPDATES_PER_EPOCH
        assert total_updates == expected_total

        # Exact ordered reviewID hash for this epoch.
        ordered_review_ids = [
            str(train_temporal.iloc[i]["reviewID"])
            for i in epoch_row_order
        ]
        order_sha = hashlib.sha256(
            "\n".join(ordered_review_ids).encode("utf-8")
        ).hexdigest()

        # Full parameter finite check once per epoch.
        bad_epoch_params = nonfinite_parameter_names(model)
        assert not bad_epoch_params, (
            f"Non-finite model parameters after epoch {epoch}: "
            f"{bad_epoch_params[:20]}"
        )

        torch.cuda.synchronize()
        elapsed = time.time() - epoch_t0
        epoch_loss = running_loss_sum / seen
        current_lr = float(scheduler.get_last_lr()[0])
        peak_alloc = int(torch.cuda.max_memory_allocated())
        peak_reserved = int(torch.cuda.max_memory_reserved())

        epoch_record = {
            "epoch": int(epoch),
            "train_loss": float(epoch_loss),
            "learning_rate_end": current_lr,
            "optimizer_updates": int(epoch_updates),
            "global_optimizer_updates": int(total_updates),
            "epoch_seconds": float(elapsed),
            "epoch_row_order_sha256": order_sha,
            "peak_gpu_allocated_bytes": peak_alloc,
            "peak_gpu_reserved_bytes": peak_reserved,
        }
        history.append(epoch_record)

        print(
            f"\nEpoch {epoch}/{HIST_EPOCHS_EXECUTED} COMPLETE | "
            f"loss={epoch_loss:.4f} | "
            f"LR={current_lr:.8f} | "
            f"{elapsed/60:.2f} min | "
            f"peak_reserved={peak_reserved/(1024**3):.2f} GiB"
        )
        print(f"Row-order SHA256: {order_sha}")

        # Atomic epoch-boundary resume checkpoint.
        save_resume_checkpoint(epoch, total_updates, history)
        print(f"✓ Atomic resume checkpoint saved after epoch {epoch}")

    # --------------------------- post-training gates --------------------------
    assert total_updates == HIST_EXECUTED_UPDATES
    assert len(history) == HIST_EPOCHS_EXECUTED
    assert scheduler.last_epoch == HIST_EXECUTED_UPDATES
    assert not nonfinite_parameter_names(model)

    FINAL_MODEL_PARAMETER_SHA = whole_model_sha256(model)

    print("\n" + "=" * 96)
    print("HISTORICAL REFIT COMPLETE — FINALISING ARTIFACT")
    print("=" * 96)
    print(f"Final model parameter SHA256: {FINAL_MODEL_PARAMETER_SHA}")

    # ------------------------ atomic artifact staging -------------------------
    STAGING_DIR.mkdir(parents=False, exist_ok=False)
    model_dir = STAGING_DIR / "model"
    model_dir.mkdir(parents=False, exist_ok=False)

    model.save_pretrained(
        model_dir,
        safe_serialization=True,
    )
    tokenizer.save_pretrained(model_dir)

    history_df = pd.DataFrame(history)
    history_path = STAGING_DIR / "training_history.csv"
    atomic_write_csv(history_df, history_path)

    registry = {
        "phase": "Phase 2A",
        "recovery_id": RECOVERY_ID,
        "stage": STAGE_ID,
        "status": "PASS",
        "origin": "Historical",
        "training_n": int(len(train_temporal)),
        "origin_fingerprint": HIST_FP,
        "model_checkpoint": MODEL_CHECKPOINT,
        "model_revision": MODEL_REVISION,
        "initial_model_parameter_sha256": INITIAL_MODEL_SHA,
        "final_model_parameter_sha256": FINAL_MODEL_PARAMETER_SHA,
        "tokenization_sha256": TOKENIZATION_SHA,
        "epochs_executed": HIST_EPOCHS_EXECUTED,
        "scheduler_horizon_epochs": HIST_SCHEDULER_HORIZON_EPOCHS,
        "physical_batch": RS1_PHYSICAL_BATCH,
        "gradient_accumulation": RS1_GRAD_ACCUM,
        "effective_batch": RS1_EFFECTIVE_BATCH,
        "optimizer_updates_per_epoch": HIST_UPDATES_PER_EPOCH,
        "executed_optimizer_updates": total_updates,
        "scheduler_total_steps": HIST_SCHEDULER_TOTAL_STEPS,
        "scheduler_steps_taken": int(scheduler.last_epoch),
        "warmup_steps": HIST_WARMUP_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_length": MAX_LENGTH,
        "max_grad_norm": MAX_GRAD_NORM,
        "precision": "FP32",
        "autocast": False,
        "grad_scaler": False,
        "optimizer_contract": optimizer_contract,
        "dataloader_num_workers": 0,
        "pin_memory": False,
        "validation_used": False,
        "heldout_accessed": False,
        "threshold_tuning": False,
        "class_weighting": False,
        "early_stopping": False,
        "resume_checkpointing": "epoch_boundary",
        "refit_contract_sha256": REFIT_CONTRACT["payload_sha256"],
        "prior_rs3s_smoke_sha256": RS3S_SMOKE_SHA256,
        "prior_rs3p_memory_contract_sha256": RS3P_MEMORY_CONTRACT_SHA256,
        "claim_bitwise_auth1_replication": False,
        "auth1_preserved_separately": True,
    }
    registry["payload_sha256"] = canonical_json_sha(registry)
    atomic_write_json(
        registry,
        STAGING_DIR / "refit_registry.json",
    )

    # File-level SHA manifest for every staged file except the manifest itself.
    file_records = []
    for path in sorted(p for p in STAGING_DIR.rglob("*") if p.is_file()):
        rel = str(path.relative_to(STAGING_DIR))
        file_records.append({
            "path": rel,
            "size_bytes": int(path.stat().st_size),
            "sha256": file_sha256(path),
        })

    artifact_manifest = {
        "recovery_id": RECOVERY_ID,
        "stage": STAGE_ID,
        "status": "PASS",
        "origin": "Historical",
        "files": file_records,
        "registry_payload_sha256": registry["payload_sha256"],
        "final_model_parameter_sha256": FINAL_MODEL_PARAMETER_SHA,
    }
    artifact_manifest["payload_sha256"] = canonical_json_sha(
        artifact_manifest
    )
    atomic_write_json(
        artifact_manifest,
        STAGING_DIR / "artifact_manifest.json",
    )

    # Finalise directory atomically on the same filesystem.
    os.replace(STAGING_DIR, FINAL_DIR)

    # Build an immediate export bundle without recompressing safetensors.
    tmp_zip = ZIP_PATH.with_suffix(".zip.tmp")
    with zipfile.ZipFile(
        tmp_zip,
        mode="w",
        compression=zipfile.ZIP_STORED,
        allowZip64=True,
    ) as zf:
        for path in sorted(p for p in FINAL_DIR.rglob("*") if p.is_file()):
            arcname = str(
                Path(FINAL_DIR.name) / path.relative_to(FINAL_DIR)
            )
            zf.write(path, arcname=arcname)

    os.replace(tmp_zip, ZIP_PATH)
    ZIP_SHA = file_sha256(ZIP_PATH)

    with open(ZIP_SHA_PATH, "w", encoding="utf-8") as f:
        f.write(f"{ZIP_SHA}  {ZIP_PATH.name}\n")
        f.flush()
        os.fsync(f.fileno())

    ready_text = (
        "PHASE 2A RS-1 HISTORICAL ARTIFACT READY\n"
        f"Artifact directory: {FINAL_DIR}\n"
        f"ZIP: {ZIP_PATH}\n"
        f"ZIP SHA256: {ZIP_SHA}\n"
        f"Registry SHA256: {registry['payload_sha256']}\n"
        f"Artifact manifest SHA256: {artifact_manifest['payload_sha256']}\n"
        f"Final model parameter SHA256: {FINAL_MODEL_PARAMETER_SHA}\n"
        "Held-out accessed during refit: False\n"
        "NEXT ACTION: persist/export the ZIP before any held-out inference.\n"
    )
    with open(READY_PATH, "w", encoding="utf-8") as f:
        f.write(ready_text)
        f.flush()
        os.fsync(f.fileno())

    # Resume checkpoint is no longer needed only after final dir + ZIP + hashes
    # have all been successfully produced.
    if RESUME_PATH.exists():
        RESUME_PATH.unlink()

    print("\n✓ RS-4H PASSED")
    print(f"✓ Final artifact directory : {FINAL_DIR}")
    print(f"✓ Export ZIP               : {ZIP_PATH}")
    print(f"✓ ZIP SHA256               : {ZIP_SHA}")
    print(f"✓ Registry SHA256          : {registry['payload_sha256']}")
    print(
        f"✓ Artifact manifest SHA256 : "
        f"{artifact_manifest['payload_sha256']}"
    )
    print("✓ Resume checkpoint removed only after successful finalisation")
    print("✓ No validation or held-out data were used")
    print("\nSTOP. Persist/export the ZIP before moving to held-out inference.")

except torch.cuda.OutOfMemoryError as e:
    # No automatic batch change is allowed. Batch 2 is the permanent contract.
    gc.collect()
    torch.cuda.empty_cache()

    failure = {
        "recovery_id": RECOVERY_ID,
        "stage": STAGE_ID,
        "status": "FAIL_OOM",
        "origin": "Historical",
        "physical_batch": RS1_PHYSICAL_BATCH,
        "gradient_accumulation": RS1_GRAD_ACCUM,
        "effective_batch": RS1_EFFECTIVE_BATCH,
        "automatic_fallback_attempted": False,
        "last_completed_epoch_checkpoint_exists": bool(RESUME_PATH.exists()),
        "message": str(e),
    }
    failure["payload_sha256"] = canonical_json_sha(failure)
    atomic_write_json(failure, FAILURE_PATH)

    raise RuntimeError(
        "Unexpected OOM under the already-proven batch-2 memory contract. "
        "Do NOT change batch size. Restart a fresh T4 runtime; the epoch-boundary "
        "resume checkpoint is retained if one had already been completed."
    ) from e

finally:
    ALLOW_MODEL_TRAINING = False


## STOP after H3

When H3 prints **✓ RS-4H PASSED**, do not run held-out inference yet.

First download and preserve:

`/kaggle/working/phase2A_RS1_HISTORICAL/phase2A_RS1_HISTORICAL_ARTIFACT.zip`

Then independently verify that downloaded ZIP's SHA-256 on Windows.

Send the final H3 summary and verified Historical ZIP SHA-256 to ChatGPT.

Only after both Full and Historical artifacts are externally preserved do we
perform the one-time frozen held-out inference for Regimes A–D.
